# Tools call in Loop

### LLM tool call in loop to get some output

In [1]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os

load_dotenv(override=True)

True

In [2]:
def show(text: str) -> None:
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI(
    base_url=os.getenv("OPENROUTER_API_URL"), api_key=os.getenv("OPENROUTER_API_KEY")
)

In [4]:
todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index"
    show(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "title": "Description",
            },
        },
        "required": ["descriptions"],
        "additionalProperties": False,
    },
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "type": "integer",
                "description": "The 1-based index of the todo to mark as complete",
                "title": "Index",
            },
            "completion_notes": {
                "type": "string",
                "description": "Notes about how you completed the todo in rich console markup",
                "title": "Completion Notes",
            },
        },
        "required": ["index", "completion_notes"],
        "additionalProperties": False,
    },
}

In [13]:
tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": mark_complete_json},
]

In [14]:
def handle_tool_call(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append(
            {
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id,
            }
        )
    return results

In [15]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(
            model="google/gemma-3-27b-it:free",
            messages=messages,
            tools=tools,  # type: ignore
            reasoning_effort="none",
        )
        if response.choices[0].finish_reason == "tool_calls":
            message = response.choices[0].message
            results = handle_tool_call(message.tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True

    show(response.choices[0].message.content)  # type: ignore

In [16]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]

In [17]:
todos, completed = [], []
loop(messages=messages)

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the distance the first train travels before the second train leaves.
Todo #3: Calculate the relative speed of the two trains.
Todo #4: Calculate the time it takes for the trains to meet after the second train leaves.
Todo #5: Calculate the meeting time.

The distance between Boston and New York is approximately 210 miles.

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the distance the first train travels before the second train leaves.
Todo #3: Calculate the relative speed of the two trains.
Todo #4: Calculate the time it takes for the trains to meet after the second train leaves.
Todo #5: Calculate the meeting time.

The first train travels for 1 hour at 60 mph, covering a distance of 60 miles.

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the distance the first train travels before the second train leaves.
Todo #3: Calculate the relative speed of the two trains.
Todo #4: Calculate the time it takes for the trains to meet after the second train leaves.
Todo #5: Calculate the meeting time.

...

The remaining distance between the trains after the first hour is 210 miles - 60 miles = 150 miles. The time it 
takes for them to meet is 150 miles / 140 mph ≈ 1.07 hours.

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the distance the first train travels before the second train leaves.
Todo #3: Calculate the relative speed of the two trains.
Todo #4: Calculate the time it takes for the trains to meet after the second train leaves.
Todo #5: Calculate the meeting time.

The second train leaves at 3:00 pm. They meet approximately 1.07 hours later, which is around 4:07 pm.

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the distance the first train travels before the second train leaves.
Todo #3: Calculate the relative speed of the two trains.
Todo #4: Calculate the time it takes for the trains to meet after the second train leaves.
Todo #5: Calculate the meeting time.

The trains meet at approximately 4:07 pm.